# Leaf Disease Detection - Model Training

Key steps:
- Data preprocessing and augmentation
- Transfer learning using MobileNetV2
- Fine-tuning for improved performance
- Model Evaluation

In [ ]:
#Import Libraries
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt

In [ ]:
#Dataset paths
train_dir ='path/to/train_dataset'
val_dir = 'path/to/validation_dataset'

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_CLASSES = 38

In [ ]:
#Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.3,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2]
)

val_datagen = ImageDataGenerator(rescale=1./255)
#Data Generators
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

In [ ]:
#Model Architecture
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs=base_model.input, outputs=outputs)


In [ ]:
#Compilation
loss_fn = tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=loss_fn,
    metrics=['accuracy']
)

model.summary()


In [ ]:
#Callbacks
early_stop = EarlyStopping(patience=3, restore_best_weights=True)

checkpoint = ModelCheckpoint(
    "best_model.h5",
    monitor='val_accuracy',
    save_best_only=True
)

lr_reduce = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=2,
    min_lr=1e-6
)

In [ ]:
#Training(Phase 1)
history = model.fit(
    train_generator,
    validation_data=val_generator,
    steps_per_epoch=500,        
    validation_steps=100,       
    epochs=10,
    callbacks=[early_stop, checkpoint, lr_reduce]
)

In [ ]:
#Fine Tuning
base_model.trainable = True

# Freeze lower layers
for layer in base_model.layers[:-30]:
    layer.trainable = False

# Recompile with lower LR
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=loss_fn,
    metrics=['accuracy']
)

history_fine = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5
)

In [ ]:
#Save the model
model.save("model.h5")

In [ ]:
#Plot Accuracy
plt.figure(figsize=(8,5))
plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.legend()
plt.title("Training vs Validation Accuracy")
plt.show()

In [ ]:
#Evaluation
loss, acc = model.evaluate(val_generator)
print(f"Validation Accuracy: {acc:.4f}")